# Module 11 — Comprehensions and Functional Tools

**Part III: Intermediate** · Level: Intermediate · Prerequisites: Modules 6, 8

## What you'll learn

- List, dictionary, and set comprehensions
- Generator expressions
- `sorted()` vs `list.sort()` with `key=`
- `map()`, `filter()`, `any()`, `all()`
- Combinatorics and chaining with `itertools`

---

## 11.1 List comprehensions

A comprehension builds a new collection from an iterable in one readable expression:

`[expression for item in iterable if condition]`

In [ ]:
squares = [n * n for n in range(5)]
print(squares)                                  # [0, 1, 4, 9, 16]

evens = [n for n in range(10) if n % 2 == 0]
print(evens)

labels = ["even" if n % 2 == 0 else "odd" for n in range(4)]   # ternary goes BEFORE `for`
print(labels)

pairs = [(r, c) for r in range(2) for c in range(3)]            # nested loops
print(pairs)

matrix = [[1, 2], [3, 4], [5, 6]]
flat = [value for row in matrix for value in row]
print(flat)

Compare with the loop it replaces:

In [ ]:
result = []
for n in range(5):
    result.append(n * n)

## 11.2 Building structured text

Comprehensions shine when you generate patterned data, such as labels or a configuration table.

In [ ]:
n = 4
horizontal = [f"{'-' * i}[]{'-' * (n - 2 - i)}" for i in range(n - 1)]
print(horizontal)

cells = [f"{col}{row}" for row in range(1, 3) for col in "ABC"]
print(cells)                                    # ['A1', 'B1', 'C1', 'A2', 'B2', 'C2']

## 11.3 Dictionary and set comprehensions

In [ ]:
sq_map = {n: n * n for n in range(5)}
print(sq_map)

evens = {n for n in range(10) if n % 2 == 0}
print(evens)

# Transform keys: reverse each key string
counts = {"100": 500, "001": 20}
reversed_keys = {key[::-1]: value for key, value in counts.items()}
print(reversed_keys)                             # {'001': 500, '100': 20}

# Invert a mapping (values must be unique and hashable)
ids = {"ada": 1, "grace": 2}
print({v: k for k, v in ids.items()})

# Filter a dict
big = {k: v for k, v in counts.items() if v > 100}
print(big)

## 11.4 Generator expressions

Replace the brackets with parentheses and you get a **lazy** generator: values are produced one at a time and never stored together. Ideal for large data or when you feed the result straight to `sum`, `max`, `any`, or `all`.

In [ ]:
gen = (n * n for n in range(5))
print(next(gen), list(gen))                      # 0 [1, 4, 9, 16]

total = sum(n * n for n in range(1_000_000))     # no million-item list is built
print(total)

import sys
print(sys.getsizeof([n for n in range(10_000)]) > sys.getsizeof(n for n in range(10_000)))

A generator can be consumed only once.

In [ ]:
g = (x for x in "abc")
print(list(g), list(g))                          # ['a', 'b', 'c'] []

## 11.5 Sorting

- `sorted(iterable)` returns a **new** list.
- `list.sort()` sorts **in place** and returns `None`.

Both accept `key=` (a function returning the sort value) and `reverse=`.

In [ ]:
nums = [3, 1, 2]
print(sorted(nums), nums)          # original unchanged
nums.sort()
print(nums)

words = ["banana", "kiwi", "apple"]
print(sorted(words, key=len))                    # shortest first
print(sorted(words, key=str.lower, reverse=True))

people = [("ada", 36), ("grace", 45), ("linus", 28)]
print(sorted(people, key=lambda p: p[1]))        # by age

# Multi-level sort: score descending, then name ascending
rows = [("ada", 90), ("bob", 90), ("cy", 85)]
print(sorted(rows, key=lambda r: (-r[1], r[0])))

Python's sort is **stable**: equal items keep their original relative order.

## 11.6 `map()`, `filter()`, `any()`, `all()`

In [ ]:
print(list(map(lambda x: x * 2, [1, 2, 3])))              # [2, 4, 6]
print(list(filter(lambda x: x % 2 == 0, [1, 2, 3, 4])))   # [2, 4]
print(list(map(str.upper, ["a", "b"])))
print(list(map(int, ["1", "2", "3"])))

# The comprehension equivalents are usually more readable
print([x * 2 for x in [1, 2, 3]])
print([x for x in [1, 2, 3, 4] if x % 2 == 0])

readings = [0.2, 0.4, 0.1]
print(any(r > 0.3 for r in readings), all(r > 0.3 for r in readings))

# Turn counts into proportions
counts = {"00": 420, "01": 25, "10": 35, "11": 520}
total = sum(counts.values())
print({k: round(v / total, 3) for k, v in counts.items()})

`map` and `filter` return lazy iterators; wrap them in `list()` to see the values.

## 11.7 `itertools`

In [ ]:
from itertools import combinations, permutations, product, chain, groupby, islice, accumulate

print(list(combinations([1, 2, 3], 2)))        # [(1, 2), (1, 3), (2, 3)]
print(list(permutations("abc", 2))[:3])
print(list(product("ab", repeat=2)))           # every 2-letter combo of a, b
print(list(chain([1, 2], [3], (4, 5))))        # flatten one level
print(list(islice(range(100), 3)))             # first three items of anything
print(list(accumulate([1, 2, 3, 4])))          # running total: [1, 3, 6, 10]

# groupby needs the data sorted by the same key
data = sorted(["apple", "avocado", "banana", "blueberry", "cherry"], key=lambda w: w[0])
for letter, group in groupby(data, key=lambda w: w[0]):
    print(letter, list(group))

## 11.8 Pythonic iteration patterns

In [ ]:
items = ["a", "b", "c"]

# Instead of: for i in range(len(items)): ...
for i, item in enumerate(items):
    print(i, item)

# Instead of: manual flags
found = any(item == "b" for item in items)

# Instead of: building strings with +=
print(", ".join(items))

# Instead of: x = x if x else default
name = ""
print(name or "default")

## Common pitfalls

- Nesting comprehensions three deep — use a normal loop.
- Using a comprehension only for its side effects (`[print(x) for x in data]`).
- Consuming a generator twice.
- Forgetting that `map`/`filter` results are lazy.
- `groupby` without sorting first.

## Practice

1. From `range(1, 101)`, build the list of squares of the numbers divisible by 7.
2. Given a list of words, build a dict mapping each word to its length, keeping only words longer than 3 letters.
3. Sort a list of `(name, age, city)` tuples by city, then by age descending.